# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via its Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This will give access to all the schema information such as record sets, fields, and columns, each referenced by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. In the Croissant metadata, each table or logical matrix is defined as a **RecordSet**, with named and `@id`-identified fields.

We will list out all record sets, showing their `@id`, their friendly name, and associated field `@id`s for reference.

In [ ]:
# List all available RecordSets by @id and their fields
record_sets = list(dataset.metadata.record_sets)

print(f"Number of RecordSets: {len(record_sets)}\n")
record_set_ids = []
record_set_fields = {}

for rs in record_sets:
    record_set_ids.append(rs['@id'])
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(unnamed)')}")
    field_ids = [f['@id'] for f in rs.get('field', [])]
    record_set_fields[rs['@id']] = field_ids
    print(f"  Field @ids: {field_ids}\n")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for exploration, using the record set and field `@id`s discovered above.

Below, you can define which record sets to extract and see the resulting DataFrame columns (these are field `@id` values, per the Croissant schema).

In [ ]:
# Select all RecordSet @ids for loading; customize if you wish to pick one or more
selected_record_sets = record_set_ids  # or select specific ones, e.g. [record_set_ids[0]]

dataframes = {}
for rs_id in selected_record_sets:
    # Load all records for each RecordSet
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded data for RecordSet {rs_id}")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"No records found for RecordSet {rs_id}\n")

## 4. Exploratory Data Analysis (EDA)
Now we process one of the loaded record sets (pick the main one containing protein information if available).

We'll select a numeric field (e.g., normalized abundance or molecular weight) by column `@id`, filter on a threshold, normalize, and group by a categorical field (e.g., sample type or accession).

⚠️ If you are unsure about field `@id`s, review the output of the prior cell for valid options.


In [ ]:
# Choose a RecordSet and numeric/categorical fields; set by their Croissant @id
if dataframes:
    main_record_set = selected_record_sets[0]  # Adjust if you want a different RecordSet
    df = dataframes[main_record_set]
    cols = df.columns.tolist()

    # Try to guess a numeric field to use; override as needed
    numeric_field_id = None
    for col in cols:
        if 'abundance' in col.lower() or 'mw' in col.lower() or 'weight' in col.lower() or 'count' in col.lower() or 'number' in col.lower():
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if not numeric_field_id:
        for col in cols:
            # Fallback: pick the first float column found
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    print(f"Selected numeric @id: {numeric_field_id}")
    threshold = df[numeric_field_id].quantile(0.75) if numeric_field_id else 10

    # Filtering
    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df
    print(f"Filtered records with {numeric_field_id} > {threshold} (top 5):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a group field; e.g. protein or sample "type"
    group_field = None
    for col in cols:
        if 'accession' in col.lower() or 'type' in col.lower() or 'name' in col.lower():
            group_field = col
            break
    if not group_field and len(cols) > 1:
        group_field = cols[1]  # fallback
    print(f"\nGrouping by: {group_field}")

    # Group by and compute mean
    if group_field in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        )
        print(f"Grouped mean of {numeric_field_id} by {group_field} (top 5):")
        print(grouped_df.head())
else:
    print('No data frames available for EDA. Check previous cells.')

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships between fields. Below, we plot histograms and optionally barplots showing grouped means.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id and not filtered_df.empty:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Barplot of grouped means (if group field is categorical and not too many groups)
    if group_field and grouped_df is not None and grouped_df[group_field].nunique() < 20:
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Insufficient data for visualization.')

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset using `mlcroissant`, examined available tables and fields via their Croissant `@id`s, extracted data, applied filtering and normalization, grouped results, and visualized key distributions. 

These steps can be adapted for any ML-Croissant dataset. For further analysis, consult the Croissant schema documentation and field metadata, always referencing fields, record sets, and other elements using their `@id` for unambiguous data access.
